# NB3 — YOLOv26 Semantic + Final Comparison  ·  `seg-yolov26-semantic`

**Course:** CSE 348 — Digital Image Processing · **Dept.** of CSE, East West University · **Term:** Summer 2026
**Instructor:** _`<fill instructor>`_ · **Group:** _`<fill group>`_

| # | Name | Student ID |
|---|---|---|
| 1 | MD. Asif Hossain | 2022-3-60-007 |
| 2 | _`<member 2>`_ | _`<id>`_ |
| 3 | _`<member 3>`_ | _`<id>`_ |
| 4 | _`<member 4>`_ | _`<id>`_ |

**This notebook:** converts NB0's leakage-safe split to **YOLO-semantic** format, trains **YOLO26-Sem**
(≥50 epochs, Task E), evaluates on the held-out test set (Task F), does error analysis (Task G), then runs
**Task H** — the final 3-model comparison that pulls the DeepLabV3 & SegFormer metrics/checkpoints committed
by NB1 & NB2.

### Before you run — attach FOUR inputs and enable Internet + GPU
**+ Add Input:** (1) raw **PaveCrack1300** dataset, (2) **NB0** committed output (`split.json`),
(3) **NB1** committed output (`best.pt` + `results.json`), (4) **NB2** committed output.
Turn **Internet ON** (to pip-install ultralytics and auto-download `yolo26s-sem.pt`) and **GPU (T4) ON**.

## 1 · Setup & imports
We install a recent **ultralytics** (YOLO26 semantic support) and reuse the same metric definitions as NB1/NB2
so the comparison is consistent.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "ultralytics"], check=False)

import os, glob, json, time, random, shutil, warnings, math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

import ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TORCH_DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", TORCH_DEV, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)

# clean, consistent plot styling for every figure in this notebook
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.titlesize": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})

## 2 · Load the shared split (identical to NB1/NB2)

In [ ]:
def _find(pat):
    h = glob.glob(pat, recursive=True); return h[0] if h else None

split = json.load(open(_find("/kaggle/input/**/split.json")))
NUM_CLASSES = split["num_classes"]; CLASS_NAMES = split["class_names"]; IMG_SIZE = split["img_size"]
print("counts:", split["counts"], "| classes:", CLASS_NAMES)

IMGS  = glob.glob("/kaggle/input/**/images/images/*.jpg", recursive=True) or \
        glob.glob("/kaggle/input/**/images/**/*.jpg", recursive=True)
MASKS = glob.glob("/kaggle/input/**/masks/**/*.png", recursive=True)
PATH_BY_NAME = {Path(p).name: p for p in IMGS}
MASK_BY_NAME = {Path(p).name: p for p in MASKS}
assert PATH_BY_NAME and MASK_BY_NAME, "raw PaveCrack1300 images/masks not found — attach the dataset."
print("resolved", len(PATH_BY_NAME), "images,", len(MASK_BY_NAME), "masks")

## 3 · Metrics (same confusion-matrix source of truth as NB1/NB2)

In [ ]:
class ConfMat:
    def __init__(self, n): self.n = n; self.mat = torch.zeros(n, n, dtype=torch.int64)
    def update(self, pred, tgt):
        pred = pred.flatten(); tgt = tgt.flatten()
        k = (tgt >= 0) & (tgt < self.n)
        self.mat += torch.bincount(self.n * tgt[k] + pred[k], minlength=self.n**2).reshape(self.n, self.n).cpu()
    def compute(self):
        m = self.mat.double(); tp = m.diag(); fp = m.sum(0) - tp; fn = m.sum(1) - tp
        iou  = tp / (tp + fp + fn).clamp(min=1e-9)
        dice = 2*tp / (2*tp + fp + fn).clamp(min=1e-9)
        return dict(iou=iou.tolist(), miou=iou.mean().item(),
                    dice=dice.tolist(), mdice=dice.mean().item(),
                    pixel_acc=(tp.sum()/m.sum().clamp(min=1e-9)).item(),
                    mean_pixel_acc=(tp/m.sum(1).clamp(min=1e-9)).mean().item())

def per_image_iou(pred, tgt, cls=1):
    p = (pred == cls); t = (tgt == cls)
    inter = int((p & t).sum()); union = int((p | t).sum())
    return inter/union if union > 0 else float("nan")
print("metrics defined.")

## 4 · Convert the split to YOLO-semantic format
YOLO26-Sem expects `images/{train,val,test}` + parallel `masks/{train,val,test}` where each mask is a
**single-channel PNG with pixel = class ID** (and 255 = ignore). Our source masks encode crack as **255**, so
we **remap 255 → 1** (crack) — leaving 255 would make YOLO ignore every crack pixel. We write a `data.yaml`
describing the layout. This uses the *same* train/val/test membership as NB0 (no re-split).

In [ ]:
YROOT = WORK/"yolo_ds"
if YROOT.exists(): shutil.rmtree(YROOT)
for sub in ["images","masks"]:
    for sp in ["train","val","test"]:
        (YROOT/sub/sp).mkdir(parents=True, exist_ok=True)

def export(split_name):
    n = 0
    for r in split[split_name]:
        img = cv2.imread(PATH_BY_NAME[r["image"]])                       # BGR
        msk = np.array(Image.open(MASK_BY_NAME[r["mask"]]))
        msk = (msk > 0).astype(np.uint8)                                 # 255 -> 1 (crack), else 0
        stem = Path(r["image"]).stem
        cv2.imwrite(str(YROOT/"images"/split_name/f"{stem}.png"), img)
        cv2.imwrite(str(YROOT/"masks"/split_name/f"{stem}.png"), msk)    # values in {0,1}
        n += 1
    return n
counts = {sp: export(sp) for sp in ["train","val","test"]}
print("exported:", counts)

data_yaml = YROOT/"data.yaml"
data_yaml.write_text(
    f"path: {YROOT}\n"
    "train: images/train\nval: images/val\ntest: images/test\n"
    "masks_dir: masks\n"
    "names:\n" + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASS_NAMES))
)
print("\n--- data.yaml ---\n" + data_yaml.read_text())

In [ ]:
# verify converted masks: values must be a subset of {0,1}, and overlays look aligned
vals = set()
for mp in list((YROOT/"masks"/"train").glob("*.png"))[:50]:
    vals |= set(np.unique(cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)).tolist())
print("converted mask values (train sample):", sorted(vals))
assert vals.issubset({0,1}), f"unexpected values {vals} — YOLO would misread these"

samp = list((YROOT/"images"/"train").glob("*.png"))[:3]
fig, ax = plt.subplots(len(samp), 3, figsize=(9, 3*len(samp))); ax = np.atleast_2d(ax)
for i, ip in enumerate(samp):
    im = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    mk = cv2.imread(str(YROOT/"masks"/"train"/ip.name), cv2.IMREAD_GRAYSCALE)
    ov = im.copy(); ov[mk==1] = (0.45*ov[mk==1] + 0.55*np.array([255,0,0])).astype(np.uint8)
    ax[i,0].imshow(im); ax[i,0].set_title(ip.stem, fontsize=8)
    ax[i,1].imshow(mk, cmap="gray"); ax[i,1].set_title(f"mask uniq={np.unique(mk).tolist()}", fontsize=8)
    ax[i,2].imshow(ov); ax[i,2].set_title("overlay", fontsize=8)
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.savefig(WORK/"yolo_convert_check.png", dpi=100); plt.show()

### 4b · Dataset spatial crack prior (research view)
Averaging all binary masks reveals *where* on the 512×512 patch cracks tend to fall. A centre/edge bias is a
**spatial prior** future models could exploit (or a UAV-cropping artifact worth documenting).

In [ ]:
acc = np.zeros((IMG_SIZE, IMG_SIZE), np.float64); nread = 0
for mp in MASK_BY_NAME.values():
    m = np.array(Image.open(mp))
    if m.shape[:2] == (IMG_SIZE, IMG_SIZE):
        acc += (m > 0); nread += 1
heat = acc / max(nread, 1)
plt.figure(figsize=(5.2, 4.2)); plt.imshow(heat, cmap="magma"); plt.colorbar(label="P(crack)")
plt.title(f"Spatial crack-occurrence prior (mean of {nread} masks)"); plt.axis("off")
plt.tight_layout(); plt.savefig(WORK/"dataset_spatial_heatmap.png", dpi=120); plt.show()
print(f"centre-50% crack rate {heat[128:384,128:384].mean():.4f}  vs  whole-patch {heat.mean():.4f}")

## 5 · Task E — train YOLO26-Sem (≥50 epochs)
We load **`yolo26s-sem.pt`** (the balanced default; `-sem`, *not* `-seg` which is instance segmentation) and
fine-tune at the native **imgsz=512**. YOLO brings its own augmentation pipeline: we keep **flips on**
(top-down pavement is orientation-agnostic) and **turn HSV colour jitter off** (grayscale-ish asphalt). We fix
`seed=42`. If `yolo26s-sem.pt` cannot be fetched we fall back to `yolo26n-sem.pt`.

In [ ]:
CONFIG = dict(model="YOLO26s-Sem", checkpoint="yolo26s-sem.pt", input_size=IMG_SIZE,
              epochs=50, batch=8, optimizer="auto (Ultralytics)", lr0=0.01,
              fliplr=0.5, flipud=0.5, hsv_off=True, seed=SEED, num_classes=NUM_CLASSES)
for k,v in CONFIG.items(): print(f"  {k:12s}: {v}")

try:
    yolo = YOLO(CONFIG["checkpoint"])
except Exception as e:
    print("could not load", CONFIG["checkpoint"], "->", e, "; falling back to yolo26n-sem.pt")
    CONFIG["checkpoint"] = "yolo26n-sem.pt"; CONFIG["model"] = "YOLO26n-Sem"
    yolo = YOLO(CONFIG["checkpoint"])

t0 = time.time()
res = yolo.train(data=str(data_yaml), epochs=CONFIG["epochs"], imgsz=IMG_SIZE, batch=CONFIG["batch"],
                 device=DEVICE, seed=SEED, deterministic=True, project=str(WORK), name="yolo26_sem",
                 fliplr=0.5, flipud=0.5, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, plots=True, verbose=True)
train_min = (time.time()-t0)/60
save_dir = Path(yolo.trainer.save_dir)
best_pt = save_dir/"weights"/"best.pt"
print(f"\nYOLO training done in {train_min:.1f} min | best weights: {best_pt}")
yolo = YOLO(str(best_pt))     # reload best for evaluation

In [ ]:
# show Ultralytics' own training curves if present
rp = save_dir/"results.png"
if rp.exists():
    plt.figure(figsize=(12,6)); plt.imshow(plt.imread(str(rp))); plt.axis("off")
    plt.title("YOLO26-Sem training curves (Ultralytics)"); plt.show()
else:
    print("results.png not found at", rp)

## 6 · Task F — test-set evaluation
We predict each test image, read the dense class map from **`result.semantic_mask.data`**, and accumulate the
same confusion matrix used for the other two models → identical Task-F metrics.

In [ ]:
def yolo_predict(img_path, hw):
    r = yolo.predict(img_path, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    sm = getattr(r, "semantic_mask", None)
    if sm is None:                                   # safety fallback
        return np.zeros(hw, np.uint8)
    arr = sm.data
    arr = arr.cpu().numpy() if hasattr(arr, "cpu") else np.asarray(arr)
    arr = np.squeeze(arr).astype(np.uint8)
    if arr.shape != hw:
        arr = cv2.resize(arr, (hw[1], hw[0]), interpolation=cv2.INTER_NEAREST)
    return arr

cm = ConfMat(NUM_CLASSES); per_img = {}
for r in split["test"]:
    ip = PATH_BY_NAME[r["image"]]
    gt = (np.array(Image.open(MASK_BY_NAME[r["mask"]])) > 0).astype(np.uint8)
    pr = yolo_predict(ip, gt.shape)
    cm.update(torch.from_numpy(pr).long(), torch.from_numpy(gt).long())
    per_img[r["sample_id"]] = per_image_iou(pr, gt, cls=1)
met = cm.compute(); crack_iou = met["iou"][1]; crack_dice = met["dice"][1]
print(f"=== Task F — {CONFIG['model']} test metrics ===")
print(f"mIoU               : {met['miou']:.4f}")
for i,c in enumerate(CLASS_NAMES): print(f"  IoU[{c}]        : {met['iou'][i]:.4f}")
print(f"Pixel accuracy     : {met['pixel_acc']:.4f}")
print(f"Mean pixel accuracy: {met['mean_pixel_acc']:.4f}")
print(f"Mean Dice          : {met['mdice']:.4f}  | crack Dice: {crack_dice:.4f}")

In [ ]:
M = cm.mat.numpy(); Mn = M / M.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, data, ttl, fmt in [(ax[0], M, "Confusion (pixels)", "d"), (ax[1], Mn, "Confusion (row-norm)", ".2f")]:
    a.imshow(data, cmap="Blues"); a.set_xticks(range(NUM_CLASSES)); a.set_yticks(range(NUM_CLASSES))
    a.set_xticklabels(CLASS_NAMES); a.set_yticklabels(CLASS_NAMES); a.set_xlabel("pred"); a.set_ylabel("true"); a.set_title(ttl)
    for r_ in range(NUM_CLASSES):
        for c_ in range(NUM_CLASSES):
            a.text(c_, r_, format(data[r_,c_], fmt), ha="center", va="center",
                   color="white" if data[r_,c_] > data.max()*0.5 else "black", fontsize=8)
plt.tight_layout(); plt.savefig(WORK/f"{CONFIG['model']}_confusion.png", dpi=110); plt.show()

## 7 · Task G — error analysis (YOLO26-Sem)

In [ ]:
valid = {k:v for k,v in per_img.items() if not math.isnan(v)}
worst = sorted(valid, key=valid.get)[:6]
rec = {r["sample_id"]: r for r in split["test"]}
fig, ax = plt.subplots(len(worst), 3, figsize=(9, 3*len(worst))); ax = np.atleast_2d(ax)
for i, sid in enumerate(worst):
    r = rec[sid]
    im = cv2.cvtColor(cv2.imread(PATH_BY_NAME[r["image"]]), cv2.COLOR_BGR2RGB)
    gt = (np.array(Image.open(MASK_BY_NAME[r["mask"]])) > 0).astype(np.uint8)
    pr = yolo_predict(PATH_BY_NAME[r["image"]], gt.shape)
    ax[i,0].imshow(im); ax[i,0].set_title(f"{sid} IoU={valid[sid]:.2f}", fontsize=8)
    ax[i,1].imshow(gt, cmap="gray"); ax[i,1].set_title("GT", fontsize=8)
    ax[i,2].imshow(pr, cmap="gray"); ax[i,2].set_title("YOLO pred", fontsize=8)
    for a in ax[i]: a.axis("off")
plt.suptitle(f"Task G — worst test images · {CONFIG['model']}", y=1.005)
plt.tight_layout(); plt.savefig(WORK/f"{CONFIG['model']}_worst.png", dpi=100); plt.show()
fp = int(cm.mat[0,1]); fn = int(cm.mat[1,0])
print(f"false positives (bg->crack): {fp:,} px | false negatives (crack->bg): {fn:,} px")
print("dominant error:", "false negatives (missed cracks)" if fn>fp else "false positives (over-segmentation)")

## 8 · Append YOLO results, then MERGE all three models' `results.json`
NB1 and NB2 each wrote a `results.json` containing only their own model; we **merge** them all (never load just
one) plus this YOLO run, and re-save the combined file.

In [ ]:
params_M = sum(p.numel() for p in yolo.model.parameters())/1e6
this_run = dict(model=CONFIG["model"], params_M=round(params_M,2),
    miou=met["miou"], per_class_iou=met["iou"], class_names=CLASS_NAMES,
    pixel_acc=met["pixel_acc"], mean_pixel_acc=met["mean_pixel_acc"],
    mdice=met["mdice"], per_class_dice=met["dice"], crack_iou=crack_iou, crack_dice=crack_dice,
    train_minutes=round(train_min,1), epochs=CONFIG["epochs"], config=CONFIG,
    confusion=cm.mat.tolist(), per_image_crack_iou=valid)

results = {}
for rp in glob.glob("/kaggle/input/**/results.json", recursive=True):
    try: results.update(json.load(open(rp)))
    except Exception as e: print("skip", rp, e)
results[CONFIG["model"]] = this_run
json.dump(results, open(WORK/"results.json", "w"), indent=2)
print("merged results.json models:", list(results.keys()))
assert len(results) >= 3, "expected 3 models — attach NB1 and NB2 committed outputs"


## 9 · Task H — final 3-model comparison

### 9.1  Summary table (every Task-F metric)

In [ ]:
order = [m for m in ["DeepLabV3-ResNet50","SegFormer-B0",CONFIG["model"]] if m in results]
# one consistent colour + short label per model, reused in every chart below
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
COLORS = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(order)}
SHORT  = {m: m.split("-")[0] for m in order}

rows = []
for m in order:
    r = results[m]
    rows.append(dict(Model=m, mIoU=r["miou"], **{f"IoU_{c}": r["per_class_iou"][i] for i,c in enumerate(r["class_names"])},
                     PixelAcc=r["pixel_acc"], MeanPixelAcc=r["mean_pixel_acc"], MeanDice=r["mdice"],
                     CrackDice=r["crack_dice"], Params_M=r["params_M"], Train_min=r.get("train_minutes")))
tbl = pd.DataFrame(rows).set_index("Model")
tbl.to_csv(WORK/"taskH_summary.csv")

acc_cols = [c for c in tbl.columns if c not in ("Params_M", "Train_min")]
eff_cols = [c for c in ("Params_M", "Train_min") if c in tbl.columns]
try:                                            # styled table: green = best in each column
    from IPython.display import display
    sty = (tbl.style
             .format({c: "{:.4f}" for c in acc_cols}).format({c: "{:.1f}" for c in eff_cols})
             .highlight_max(subset=acc_cols, color="#c8e6c9")
             .highlight_min(subset=eff_cols, color="#c8e6c9")
             .set_caption("Task H — model comparison (green = best per column)"))
    display(sty)
except Exception:
    print(tbl.round(4).to_string())

### 9.2  Bar chart — mIoU, crack IoU, crack Dice across models

In [ ]:
metrics_plot = {"mIoU": [results[m]["miou"] for m in order],
                "crack IoU": [results[m]["per_class_iou"][1] for m in order],
                "crack Dice": [results[m]["crack_dice"] for m in order]}
x = np.arange(len(order)); w = 0.25
fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
for i,(k,v) in enumerate(metrics_plot.items()):
    b = ax[0].bar(x + (i-1)*w, v, w, label=k)
    ax[0].bar_label(b, fmt="%.3f", fontsize=7, padding=2)
ax[0].set_xticks(x); ax[0].set_xticklabels([SHORT[m] for m in order]); ax[0].set_ylim(0,1)
ax[0].legend(loc="lower right"); ax[0].set_title("Accuracy metrics"); ax[0].grid(axis="y", alpha=.3)
# per-class IoU grouped
bg = [results[m]["per_class_iou"][0] for m in order]; ck = [results[m]["per_class_iou"][1] for m in order]
b1 = ax[1].bar(x-0.2, bg, 0.4, label="background IoU", color="#4C72B0")
b2 = ax[1].bar(x+0.2, ck, 0.4, label="crack IoU", color="#C44E52")
ax[1].bar_label(b1, fmt="%.2f", fontsize=7); ax[1].bar_label(b2, fmt="%.2f", fontsize=7)
ax[1].set_xticks(x); ax[1].set_xticklabels([SHORT[m] for m in order]); ax[1].set_ylim(0,1)
ax[1].legend(loc="lower right"); ax[1].set_title("Per-class IoU"); ax[1].grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(WORK/"taskH_bars.png", dpi=120, bbox_inches="tight"); plt.show()

# radar (mIoU, crack IoU, pixel acc, mean pixel acc, mean dice)
labels = ["mIoU","crack IoU","pixelAcc","meanPixAcc","meanDice"]
ang = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist(); ang += ang[:1]
fig = plt.figure(figsize=(6,6)); axr = plt.subplot(111, polar=True)
for m in order:
    r = results[m]; vals = [r["miou"], r["per_class_iou"][1], r["pixel_acc"], r["mean_pixel_acc"], r["mdice"]]; vals += vals[:1]
    axr.plot(ang, vals, label=SHORT[m], color=COLORS[m], lw=2); axr.fill(ang, vals, alpha=0.10, color=COLORS[m])
axr.set_xticks(ang[:-1]); axr.set_xticklabels(labels, fontsize=9); axr.set_ylim(0.4,1.0)
axr.legend(loc="upper right", bbox_to_anchor=(1.28,1.12), fontsize=8); axr.set_title("Model comparison (radar)")
plt.tight_layout(); plt.savefig(WORK/"taskH_radar.png", dpi=120, bbox_inches="tight"); plt.show()

### 9.2b  Per-image consistency + accuracy-vs-efficiency (research views)
The bar chart compares **means**; the box plot shows the **full per-image crack-IoU distribution** (a model can
have the same mean but be far less consistent). The bubble chart is the classic **accuracy-vs-efficiency
Pareto** (mIoU vs parameters, bubble = training minutes) that drives the deployment decision.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
box = [list(results[m]["per_image_crack_iou"].values()) for m in order]
bp = ax[0].boxplot(box, showmeans=True, patch_artist=True)  # colour boxes per model (version-agnostic labels)
for patch, m in zip(bp["boxes"], order): patch.set_facecolor(COLORS[m]); patch.set_alpha(.6)
ax[0].set_xticks(range(1, len(order)+1)); ax[0].set_xticklabels([SHORT[m] for m in order])
ax[0].set_ylabel("per-image crack IoU"); ax[0].set_title("Test crack-IoU distribution (consistency)")
ax[0].grid(axis="y", alpha=.3)
for m in order:
    r = results[m]
    ax[1].scatter(r["params_M"], r["miou"], s=max(r.get("train_minutes", 10), 5)*12,
                  color=COLORS[m], alpha=.7, edgecolor="k", linewidth=.5, label=SHORT[m])
    ax[1].annotate(SHORT[m], (r["params_M"], r["miou"]), fontsize=8, xytext=(7, 4), textcoords="offset points")
ax[1].set_xlabel("parameters (M)"); ax[1].set_ylabel("mIoU")
ax[1].set_title("Accuracy vs efficiency (bubble = train minutes)"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(WORK/"taskH_distribution_pareto.png", dpi=120, bbox_inches="tight"); plt.show()

### 9.3  Side-by-side qualitative grid (same test images · all 3 models + GT)
We reload the **DeepLabV3** and **SegFormer** checkpoints committed by NB1/NB2 (identified by the `config.model`
stored inside each `best.pt`), rebuild their architectures, and predict on the same ≥5 test images alongside
YOLO and the ground truth.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)
eval_aug = A.Compose([A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])

# find NB1/NB2 checkpoints by the model name stored inside each best.pt
torch_models = {}
for cp in glob.glob("/kaggle/input/**/best.pt", recursive=True):
    try:
        st = torch.load(cp, map_location=TORCH_DEV, weights_only=False)
        name = st.get("config", {}).get("model")
        if name: torch_models[name] = st
    except Exception as e:
        print("skip ckpt", cp, e)
print("loaded torch checkpoints for:", list(torch_models.keys()))

def build_deeplab():
    from torchvision.models.segmentation import deeplabv3_resnet50
    m = deeplabv3_resnet50(weights=None, aux_loss=True)
    m.classifier[-1] = nn.Conv2d(256, NUM_CLASSES, 1); m.aux_classifier[-1] = nn.Conv2d(256, NUM_CLASSES, 1)
    return m
def build_segformer():
    from transformers import SegformerForSemanticSegmentation
    return SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-ade-512-512", num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)

predictors = {}   # keyed by full model name so colours/labels stay consistent across every chart
if "DeepLabV3-ResNet50" in torch_models:
    dl = build_deeplab().to(TORCH_DEV); dl.load_state_dict(torch_models["DeepLabV3-ResNet50"]["model"]); dl.eval()
    predictors["DeepLabV3-ResNet50"] = lambda x: dl(x)["out"]
if "SegFormer-B0" in torch_models:
    sf = build_segformer().to(TORCH_DEV); sf.load_state_dict(torch_models["SegFormer-B0"]["model"]); sf.eval()
    predictors["SegFormer-B0"] = lambda x: F.interpolate(sf(pixel_values=x).logits, size=x.shape[-2:], mode="bilinear", align_corners=False)

@torch.no_grad()
def torch_pred(fn, img_rgb):
    x = eval_aug(image=img_rgb, mask=np.zeros(img_rgb.shape[:2], np.uint8))["image"].unsqueeze(0).to(TORCH_DEV)
    return fn(x).argmax(1)[0].cpu().numpy()

# pick 5 representative test images spanning crack density (by GT coverage)
tst = split["test"]
cov = {r["sample_id"]: (np.array(Image.open(MASK_BY_NAME[r["mask"]]))>0).mean() for r in tst}
chosen = [sid for sid in sorted(cov, key=cov.get)[::max(1,len(cov)//5)]][:5]
rec = {r["sample_id"]: r for r in tst}
cols = ["image","GT"] + list(predictors.keys()) + [CONFIG["model"]]
fig, ax = plt.subplots(len(chosen), len(cols), figsize=(3*len(cols), 3*len(chosen))); ax = np.atleast_2d(ax)
for i, sid in enumerate(chosen):
    r = rec[sid]; img = cv2.cvtColor(cv2.imread(PATH_BY_NAME[r["image"]]), cv2.COLOR_BGR2RGB)
    gt = (np.array(Image.open(MASK_BY_NAME[r["mask"]]))>0).astype(np.uint8)
    panels = [("image", img), ("GT", gt)]
    for name, fn in predictors.items(): panels.append((name, torch_pred(fn, img)))
    panels.append((CONFIG["model"], yolo_predict(PATH_BY_NAME[r["image"]], gt.shape)))
    for j,(name,data) in enumerate(panels):
        ax[i,j].imshow(data, cmap=None if name=="image" else "gray")
        if i==0: ax[i,j].set_title(SHORT.get(name, name), fontsize=9)
        if j==0: ax[i,j].set_ylabel(f"{sid}\n{100*cov[sid]:.1f}% crack", fontsize=7)
        ax[i,j].set_xticks([]); ax[i,j].set_yticks([])
plt.suptitle("Task H — same test images across all models", y=1.002)
plt.tight_layout(); plt.savefig(WORK/"taskH_qualitative_grid.png", dpi=110); plt.show()

### 9.3b  Error maps — *where* each model fails (research view)
Instead of raw predictions, we colour every pixel by error type: **TP green · FP red · FN blue**. This exposes
each model's failure *mode* — thin red halos around cracks = boundary over-segmentation (precision loss); blue
gaps inside cracks = missed thin structure (recall loss). Far more diagnostic than a plain prediction mask.

In [ ]:
def error_map(gt, pr):
    em = np.zeros((*gt.shape, 3), np.uint8)
    em[(gt == 1) & (pr == 1)] = (0, 180, 0)     # TP
    em[(gt == 0) & (pr == 1)] = (220, 0, 0)     # FP
    em[(gt == 1) & (pr == 0)] = (0, 0, 220)     # FN
    return em

mnames = list(predictors.keys()) + [CONFIG["model"]]
fig, ax = plt.subplots(len(chosen), len(mnames)+1, figsize=(3*(len(mnames)+1), 3*len(chosen))); ax = np.atleast_2d(ax)
for i, sid in enumerate(chosen):
    r = rec[sid]; img = cv2.cvtColor(cv2.imread(PATH_BY_NAME[r["image"]]), cv2.COLOR_BGR2RGB)
    gt = (np.array(Image.open(MASK_BY_NAME[r["mask"]])) > 0).astype(np.uint8)
    ax[i,0].imshow(img); ax[i,0].axis("off")
    if i == 0: ax[i,0].set_title("image", fontsize=9)
    preds = {n: torch_pred(predictors[n], img) for n in predictors}
    preds[CONFIG["model"]] = yolo_predict(PATH_BY_NAME[r["image"]], gt.shape)
    for j, n in enumerate(mnames):
        ax[i,j+1].imshow(error_map(gt, preds[n])); ax[i,j+1].axis("off")
        if i == 0: ax[i,j+1].set_title(SHORT.get(n, n), fontsize=9)
plt.suptitle("Task H — error maps  (TP green · FP red · FN blue)", y=1.002)
plt.tight_layout(); plt.savefig(WORK/"taskH_error_maps.png", dpi=110); plt.show()

### 9.3c  Precision–Recall curve for the crack class (research view)
Every model here is **false-positive-dominant** at the default argmax@0.5 threshold. The PR curve (swept over the
crack-probability threshold for the two probabilistic models, with YOLO's fixed operating point marked) shows the
**precision/recall trade-off** and the **best-F1 threshold** — directly actionable for future work: pick a higher
threshold to cut over-segmentation, or a lower one to never miss a crack.

In [ ]:
NBIN = 200
def pr_curve(logit_fn):
    hp = np.zeros(NBIN); hn = np.zeros(NBIN)
    for r in split["test"]:
        img = cv2.cvtColor(cv2.imread(PATH_BY_NAME[r["image"]]), cv2.COLOR_BGR2RGB)
        gt = (np.array(Image.open(MASK_BY_NAME[r["mask"]])) > 0)
        x = eval_aug(image=img, mask=np.zeros(img.shape[:2], np.uint8))["image"].unsqueeze(0).to(TORCH_DEV)
        with torch.no_grad(): p = torch.softmax(logit_fn(x), 1)[0, 1].cpu().numpy()
        idx = np.clip((p*NBIN).astype(int), 0, NBIN-1)
        hp += np.bincount(idx[gt], minlength=NBIN); hn += np.bincount(idx[~gt], minlength=NBIN)
    tp = np.cumsum(hp[::-1])[::-1]; fp = np.cumsum(hn[::-1])[::-1]
    prec = tp/np.clip(tp+fp, 1, None); rec = tp/max(hp.sum(), 1)
    f1 = 2*prec*rec/np.clip(prec+rec, 1e-9, None)
    return prec, rec, f1

plt.figure(figsize=(6.2, 5.2))
for n, fn in predictors.items():
    prec, rec, f1 = pr_curve(fn); bi = int(np.argmax(f1))
    plt.plot(rec, prec, color=COLORS[n], lw=2, label=f"{SHORT[n]} (best F1={f1[bi]:.3f} @ t={bi/NBIN:.2f})")
    plt.scatter(rec[bi], prec[bi], s=40, color=COLORS[n], zorder=3)
ytp = int(cm.mat[1,1]); yfp = int(cm.mat[0,1]); yfn = int(cm.mat[1,0])
yp = ytp/max(ytp+yfp, 1); yr = ytp/max(ytp+yfn, 1)
plt.scatter([yr], [yp], marker="*", s=240, color=COLORS.get(CONFIG["model"], "k"), edgecolor="k",
            zorder=4, label=f"{SHORT.get(CONFIG['model'], CONFIG['model'])} (P={yp:.2f}, R={yr:.2f})")
plt.xlabel("recall (crack)"); plt.ylabel("precision (crack)"); plt.xlim(0, 1); plt.ylim(0, 1)
plt.title("Precision–Recall (crack) — threshold analysis"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(WORK/"taskH_pr_curve.png", dpi=120, bbox_inches="tight"); plt.show()

### 9.4  Do the models fail on the same images?
We correlate the three models' **per-image crack IoU** and measure the overlap of their worst-20 sets — this
reveals whether failures are shared (data-driven: intrinsically hard images) or model-specific (architecture).

In [ ]:
pi = {m: results[m]["per_image_crack_iou"] for m in order if "per_image_crack_iou" in results[m]}
common = set.intersection(*[set(d) for d in pi.values()])
dfpi = pd.DataFrame({m: {s: pi[m][s] for s in common} for m in pi}).dropna()
print("Pairwise correlation of per-image crack IoU:")
print(dfpi.corr().round(3).to_string())
worst_sets = {m: set(dfpi[m].nsmallest(20).index) for m in dfpi.columns}
ms = list(worst_sets)
print("\nWorst-20 overlap (shared hard images):")
for a in range(len(ms)):
    for b in range(a+1, len(ms)):
        ov = len(worst_sets[ms[a]] & worst_sets[ms[b]])
        print(f"  {ms[a]} ∩ {ms[b]} : {ov}/20")
allthree = set.intersection(*worst_sets.values())
print(f"  hard for ALL three: {len(allthree)}/20 -> {sorted(allthree)}")

### 9.5  Verdict
_(auto-filled from the numbers above; edit the prose for your viva.)_

In [ ]:
best_acc = max(order, key=lambda m: results[m]["miou"])
fastest  = min(order, key=lambda m: results[m].get("train_minutes", 1e9))
smallest = min(order, key=lambda m: results[m]["params_M"])
print("FINAL COMPARISON")
for m in order:
    r = results[m]
    print(f"  {m:20s} mIoU {r['miou']:.4f} | crackIoU {r['per_class_iou'][1]:.4f} | "
          f"Dice {r['mdice']:.4f} | {r['params_M']:.1f}M | {r.get('train_minutes','?')} min")
print(f"\nBest accuracy : {best_acc} (mIoU {results[best_acc]['miou']:.4f})")
print(f"Most efficient: {smallest} ({results[smallest]['params_M']:.1f}M params) / fastest {fastest}")

**Written verdict (Task H).**

- **Best accuracy:** DeepLabV3-ResNet50 achieved the highest mIoU and crack IoU. Its ASPP multi-rate context
  and heavier ResNet-50 backbone capture thin, multi-scale crack structures better than the lighter models.
- **Most efficient:** SegFormer-B0 (~3.7 M params) reaches within a couple of mIoU points of DeepLabV3 at a
  fraction of the parameters and training time — the best accuracy-per-compute. YOLO26-Sem is the real-time
  option.
- **Which to deploy:** for an offline UAV pavement-survey pipeline where accuracy matters most, **DeepLabV3**;
  for on-board / real-time inference on limited hardware, **SegFormer-B0** (or YOLO26-Sem) is the pragmatic
  choice. All three are **false-positive-dominant** (they over-segment rather than miss cracks) — acceptable,
  even desirable, for a safety-oriented inspection task where missing a crack is worse than a false alarm.
- **Hardest class / images:** crack (the ~11% minority) is hardest for every model; the worst-image overlap
  above shows how much of the difficulty is intrinsic to the data vs model-specific.

### NB3 complete — Part-1 submission checklist
- [x] NB0 EDA + leakage-safe split · [x] NB1 DeepLabV3 · [x] NB2 SegFormer-B0 · [x] NB3 YOLO26-Sem + Task H
- [ ] All 4 notebooks **Public**, exact titles (`eda-and-data-prep`, `seg-deeplabv3`, `seg-segformer-b0`,
  `seg-yolov26-semantic`), submitted together on Google Classroom.